# Data assets

In [1]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [1]:
!sh ./helper_code/create_a_blob_store.sh

/c/Users/dgouwy/Documents/devoProjects/azure-machine-learning/az-ml/03_data-in-az-ml
storage account with name dg123 already exists
... start deleting
... Start creating storage account
... Get the account key
... Create a container
... Upload a CSV
... Create an SAS token
... Create a new .env file
Done!


## Setup

In [3]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
import os
import dotenv

dotenv.load_dotenv(override=True)

ml_client = MLClient(
     credential         = DefaultAzureCredential()
    ,subscription_id    = os.environ["AZURE_SUBSCRIPTION_ID"]
    ,resource_group_name= os.environ["AZURE_RESOURCE_GROUP"]
    ,workspace_name     = os.environ["AZURE_ML"]
)

compute_name = "diabetes-cluster"

def list_az_ml(obj):
    return [o for o in obj.list()]
        

Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


In [3]:
list(map(lambda x: x.name, list_az_ml(ml_client.data)))

[]

In [4]:
from azure.ai.ml.entities import AmlCompute

compute_name = "diabetes-cluster"

aml_compute = AmlCompute(
     name                           = compute_name
    ,size                           = "Standard_DS11_v2"
    ,min_instances                  = 0
    ,max_instances                  = 1
    ,idle_time_before_scale_down    = 30
)

try:
    poller = ml_client.begin_create_or_update(aml_compute)
    while not poller.done:
        print(f"Setting up compute: {compute_name}.", end="\r")
    print(f"Compute {compute_name} created")
except Exception as e:
    raise e

Compute diabetes-cluster created


## Create a URI file

Note that you need to specify the ```AssetTypes``` from ```from azure.ai.ml.constants import AssetTypes```. Specify ```URI_FILE``` specifically. Also when using the data in the job you better specify the same thing. Note that you need to add an inputs to the job. There you can just point to the asset name and version. 

In [5]:

file_path = "../../data/diabetes-data/diabetes.csv"
name = "diabetes-data"

my_data = Data(
    path=file_path,
    type=AssetTypes.URI_FILE,
    description="diabetes data in csv format",
    name=name,
    version="0.1"
)

ml_client.data.create_or_update(my_data)

Uploading diabetes.csv (< 1 MB): 100%|##########| 528k/528k [00:00<00:00, 5.74MB/s]




Data({'path': 'azureml://subscriptions/e46b15c2-60e4-419f-a352-ee019a3b0daf/resourcegroups/dg-associate-data-science/workspaces/aml-dg-associate-data-science/datastores/workspaceblobstore/paths/LocalUpload/55b013eca413aa16a425d4674a54a4ecadf28a7027e8272f1a0fdbf15ca32190/diabetes.csv', 'skip_validation': False, 'mltable_schema_url': None, 'referenced_uris': None, 'type': 'uri_file', 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'diabetes-data', 'description': 'diabetes data in csv format', 'tags': {}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/e46b15c2-60e4-419f-a352-ee019a3b0daf/resourceGroups/dg-associate-data-science/providers/Microsoft.MachineLearningServices/workspaces/aml-dg-associate-data-science/data/diabetes-data/versions/0.1', 'Resource__source_path': '', 'base_path': 'c:\\Users\\dgouwy\\Documents\\devoProjects\\azure-machine-learning\\az-ml\\03_data-in-az-ml', 'creation_context': <azure.ai.ml.entities._system

In [6]:
list(map(lambda x: x.name, list_az_ml(ml_client.data)))

['diabetes-data']

Write code to local dir

In [25]:
%%writefile code/data-asset-file-script.py

import argparse
import pandas as pd

parser = argparse.ArgumentParser()
parser.add_argument("--data-asset", type=str)
args = parser.parse_args()

df = pd.read_csv(args.data_asset)
print(df.head())

Overwriting code/data-asset-file-script.py


Check out if data asset is being used

In [ ]:
from azure.ai.ml import command, Input
from azure.ai.ml.constants import AssetTypes
import time

data_asset = "diabetes-data"
version = "0.1"

job = command(
     code               = "./code"
    ,command            = f"python -m pip install azure-ai-ml && \
                            python -m pip install azure-identity && \
                            python data-asset-file-script.py --data-asset ${{inputs.my_dataset}}"
    ,inputs = {
        "my_dataset": Input(type=AssetTypes.URI_FILE, path="azureml:diabetes-data:0.1")
    }
    ,environment        = "AzureML-sklearn-1.5@latest"
    ,compute            = compute_name
    ,display_name       = "diabetes-train-data-discovery"
    ,experiment_name    = "diabetes-training"
)

while True:
    time.sleep(1)
    compute_state = ml_client.compute.get(compute_name).provisioning_state.lower()
    if compute_state == "succeeded":
        returned_job = ml_client.create_or_update(job)
        break 
print(returned_job.studio_url)


Uploading code (0.0 MBs): 100%|##########| 208/208 [00:00<00:00, 8677.55it/s]


Use of {} for parameters is deprecated, instead use ${{}}.


https://ml.azure.com/runs/sleepy_screw_0hm1t718j5?wsid=/subscriptions/e46b15c2-60e4-419f-a352-ee019a3b0daf/resourcegroups/dg-associate-data-science/workspaces/aml-dg-associate-data-science&tid=d018aec4-2b2b-4c66-9939-2c96877e6bf1


## Create a URI folder

In [10]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

folder_path = "../../data/diabetes-data"

my_data = Data(
    path=folder_path,
    type=AssetTypes.URI_FOLDER,
    description="Uploading a URI folder",
    name="diabetes-data-files",
    version='2'
)

ml_client.data.create_or_update(my_data)

Data({'path': 'azureml://subscriptions/e46b15c2-60e4-419f-a352-ee019a3b0daf/resourcegroups/dg-associate-data-science/workspaces/aml-dg-associate-data-science/datastores/workspaceblobstore/paths/LocalUpload/38a26b56d2c4d70e3c285c51e67c1874783dd15cf53623047b76421f171e8933/diabetes-data/', 'skip_validation': False, 'mltable_schema_url': None, 'referenced_uris': None, 'type': 'uri_folder', 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'diabetes-data-files', 'description': 'Uploading a URI folder', 'tags': {}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/e46b15c2-60e4-419f-a352-ee019a3b0daf/resourceGroups/dg-associate-data-science/providers/Microsoft.MachineLearningServices/workspaces/aml-dg-associate-data-science/data/diabetes-data-files/versions/2', 'Resource__source_path': '', 'base_path': 'c:\\Users\\dgouwy\\Documents\\devoProjects\\azure-machine-learning\\az-ml\\03_data-in-az-ml', 'creation_context': <azure.ai.ml.entitie

In [11]:
%%writefile code/data-asset-folder-script.py

import argparse
import glob
import pandas as pd

parser = argparse.ArgumentParser()
parser.add_argument("--input-data", type=str)

args = parser.parse_args()

data_path = args.input_data
all_files = glob.glob(data_path + "/*.csv")

df = pd.concat((pd.read_csv(f) for f in all_files), sort=False)

Overwriting code/data-asset-folder-script.py


In [12]:
from azure.ai.ml import command, Input
from azure.ai.ml.constants import AssetTypes
import time

data_asset = "diabetes-data-files"
version = "2"


job = command(
     code               = "./code"
    ,command            = f"python -m pip install azure-ai-ml && \
                            python -m pip install azure-identity && \
                            python data-asset-folder-script.py --input-data ${{inputs.my_dataset}}"
    ,inputs = {
        "my_dataset": Input(type=AssetTypes.URI_FOLDER, path=f"azureml:{data_asset}:{version}")
    }
    ,environment        = "AzureML-sklearn-1.5@latest"
    ,compute            = compute_name
    ,display_name       = "diabetes-train-data-discovery-uri-folder"
    ,experiment_name    = "diabetes-training"
)

while True:
    time.sleep(1)
    compute_state = ml_client.compute.get(compute_name).provisioning_state.lower()
    if compute_state == "succeeded":
        returned_job = ml_client.create_or_update(job)
        break 
print(returned_job.studio_url)

Uploading code (0.0 MBs): 100%|##########| 775/775 [00:00<00:00, 1175.62it/s]


Use of {} for parameters is deprecated, instead use ${{}}.


https://ml.azure.com/runs/red_basin_tz02dxxjzn?wsid=/subscriptions/e46b15c2-60e4-419f-a352-ee019a3b0daf/resourcegroups/dg-associate-data-science/workspaces/aml-dg-associate-data-science&tid=d018aec4-2b2b-4c66-9939-2c96877e6bf1


## Create an MLTable data asset

In [6]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

folder_path = "../../data/diabetes-data"

my_data = Data(
    path=folder_path,
    type=AssetTypes.MLTABLE,
    description="Uploading an MLTable",
    name="diabetes-data-ml-table",
    version='0'
)

ml_client.data.create_or_update(my_data)

Data({'path': 'azureml://subscriptions/e46b15c2-60e4-419f-a352-ee019a3b0daf/resourcegroups/dg-associate-data-science/workspaces/aml-dg-associate-data-science/datastores/workspaceblobstore/paths/LocalUpload/38a26b56d2c4d70e3c285c51e67c1874783dd15cf53623047b76421f171e8933/diabetes-data/', 'skip_validation': False, 'mltable_schema_url': None, 'referenced_uris': ['./diabetes.csv'], 'type': 'mltable', 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'diabetes-data-ml-table', 'description': 'Uploading an MLTable', 'tags': {}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/e46b15c2-60e4-419f-a352-ee019a3b0daf/resourceGroups/dg-associate-data-science/providers/Microsoft.MachineLearningServices/workspaces/aml-dg-associate-data-science/data/diabetes-data-ml-table/versions/0', 'Resource__source_path': '', 'base_path': 'c:\\Users\\dgouwy\\Documents\\devoProjects\\azure-machine-learning\\az-ml\\03_data-in-az-ml', 'creation_context': <azur

In [13]:
%%writefile code/data-asset-ml-table-script.py

import argparse
import mltable
import pandas

parser = argparse.ArgumentParser()
parser.add_argument("--input-data", type=str)
args = parser.parse_args()

tbl = mltable.load(args.input_data)
df = tbl.to_pandas_dataframe()

print(df.head(10))

Overwriting code/data-asset-ml-table-script.py


In [16]:
from azure.ai.ml import command, Input
from azure.ai.ml.constants import AssetTypes
import time

job = command(
     code               = "./code"
    ,command            = f"python -m pip install azure-ai-ml && \
                            python -m pip install azure-identity && \
                            python -m pip install mltable && \
                            python data-asset-ml-table-script.py --input-data ${{inputs.my_dataset}}"
    ,inputs = {
        "my_dataset": Input(type=AssetTypes.MLTABLE, path="azureml:diabetes-data-ml-table:0")
    }
    ,environment        = "AzureML-sklearn-1.5@latest"
    ,compute            = compute_name
    ,display_name       = "diabetes-train-data-discovery-ml-table"
    ,experiment_name    = "diabetes-training"
)

while True:
    time.sleep(1)
    compute_state = ml_client.compute.get(compute_name).provisioning_state.lower()
    if compute_state == "succeeded":
        returned_job = ml_client.create_or_update(job)
        break 
print(returned_job.studio_url)

Use of {} for parameters is deprecated, instead use ${{}}.


https://ml.azure.com/runs/quiet_nut_9l4g24htjg?wsid=/subscriptions/e46b15c2-60e4-419f-a352-ee019a3b0daf/resourcegroups/dg-associate-data-science/workspaces/aml-dg-associate-data-science&tid=d018aec4-2b2b-4c66-9939-2c96877e6bf1


## Cleanup

In [ ]:
try:
    poller = ml_client.compute.begin_delete(compute_name)
    while not poller.done:
        print(f"Deleting compute: {compute_name}.", end="\r")
    print({f"Compute {compute_name} deleted"})
except Exception as e:
    raise e